In [2]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

root = Path("/Users/benjamin/Workspace_GAMA/MAELIA/includes/terrainTest")

ilots_path = root / "modeleAgricole/ilots/dansZone/ilots.shp"
sols_path = root / "modeleCommun/typesDeSol/typeDeSolParZH.shp"

ilots = gpd.read_file(ilots_path)
sols = gpd.read_file(sols_path)

# =========================
# 1) Correspondance climat
# =========================

climat_par_contexte = {
    "beauce": 13218,
    "oceanique": 13761,
    "sudouest": 4756,
}

def extraire_contexte(id_ilot):
    x = str(id_ilot).lower()
    if x.startswith("copie_"):
        x = x.replace("copie_", "", 1)

    if x.startswith("beauce"):
        return "beauce"
    if x.startswith("oceanique") or x.startswith("océanique"):
        return "oceanique"
    if x.startswith("sudouest") or x.startswith("sud-ouest"):
        return "sudouest"
    return None

# =========================
# 2) Correspondance sol
# =========================

sols_simple = sols[
    ["ID_SOL", "ZONE_PEDO", "INFO_SOL"]
].drop_duplicates()

# =========================
# 3) Table finale sans fertilisation
# =========================

df = ilots.drop(columns="geometry", errors="ignore").copy()

df["CONTEXTE_CLIMATIQUE"] = df["ID_ILOT"].apply(extraire_contexte)
df["ID_PDG_METEO"] = df["CONTEXTE_CLIMATIQUE"].map(climat_par_contexte)

df = df.merge(sols_simple, on="ID_SOL", how="left")

colonnes = [
    "ID_ILOT",
    "CONTEXTE_CLIMATIQUE",
    "ID_PDG_METEO",
    "ID_SOL",
    "ZONE_PEDO",
    "INFO_SOL",
    "ID_ZH",
]

df_final = df[colonnes].sort_values(
    ["CONTEXTE_CLIMATIQUE", "ID_SOL", "ID_ILOT"]
)

display(df_final)

# =========================
# 4) Résumé des combinaisons sol × climat
# =========================

resume = (
    df_final
    .groupby(
        [
            "CONTEXTE_CLIMATIQUE",
            "ID_PDG_METEO",
            "ID_SOL",
            "ZONE_PEDO",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="nombre_ilots")
    .sort_values(
        [
            "CONTEXTE_CLIMATIQUE",
            "ID_SOL",
        ]
    )
)

display(resume)

print("Nombre de climats :", df_final["CONTEXTE_CLIMATIQUE"].nunique(dropna=True))
print("Nombre de météos ID_PDG :", df_final["ID_PDG_METEO"].nunique(dropna=True))
print("Nombre de sols :", df_final["ID_SOL"].nunique(dropna=True))
print(
    "Nombre de combinaisons sol × climat :",
    df_final[["CONTEXTE_CLIMATIQUE", "ID_SOL"]].drop_duplicates().shape[0]
)

,ID_ILOT,CONTEXTE_CLIMATIQUE,ID_PDG_METEO,ID_SOL,ZONE_PEDO,INFO_SOL,ID_ZH
9,beauce_19,beauce,13218,18960_27491,limoneux,LUVISOL REDOXISOL sablo-limoneux a galets roul...,1
33,beauce_20,beauce,13218,18960_27491,limoneux,LUVISOL REDOXISOL sablo-limoneux a galets roul...,1
45,beauce_21,beauce,13218,18960_27491,limoneux,LUVISOL REDOXISOL sablo-limoneux a galets roul...,1
27,beauce_22,beauce,13218,18960_27491,limoneux,LUVISOL REDOXISOL sablo-limoneux a galets roul...,1
3,beauce_23,beauce,13218,18960_27491,limoneux,LUVISOL REDOXISOL sablo-limoneux a galets roul...,1
...,...,...,...,...,...,...,...
94,sudouest_38,sudouest,4756,330151_330489_1,argilo-calcaire,ctx_arg,1
76,sudouest_39,sudouest,4756,330151_330489_1,argilo-calcaire,ctx_arg,1
88,sudouest_40,sudouest,4756,330151_330489_1,argilo-calcaire,ctx_arg,1
118,sudouest_7,sudouest,4756,330151_330489_1,argilo-calcaire,ctx_arg,1


,CONTEXTE_CLIMATIQUE,ID_PDG_METEO,ID_SOL,ZONE_PEDO,nombre_ilots
0,beauce,13218,18960_27491,limoneux,16
1,beauce,13218,330104_330367_1,limono-sableux,18
2,beauce,13218,330151_330489_1,argilo-calcaire,17
3,oceanique,13761,18960_27491,limoneux,9
4,oceanique,13761,330104_330367_1,limono-sableux,9
5,oceanique,13761,330151_330489_1,argilo-calcaire,9
6,sudouest,4756,18960_27491,limoneux,17
7,sudouest,4756,330104_330367_1,limono-sableux,17
8,sudouest,4756,330151_330489_1,argilo-calcaire,18


Nombre de climats : 3
Nombre de météos ID_PDG : 3
Nombre de sols : 3
Nombre de combinaisons sol × climat : 9
